[Reference](https://medium.com/@Rohan_Dutt/10-pyspark-optimization-patterns-that-work-without-advanced-tuning-for-data-engineers-3e497bd8e186)

# 1. Avoid the .collect() Trap

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

import pandas as pd

spark = SparkSession.builder.appName("collect_demo").getOrCreate()

# Create a 5GB DataFrame (1B rows)
data = spark.range(0, 1_000_000_000).withColumn("value", F.rand() * 1000)

# BAD: Crashes driver with OOM
# all_data = data.collect()  # Never do this on large data

# GOOD: Sample with .take()
sample = data.take(10)  # Pulls only 10 rows to driver
print(f"Sample rows: {sample}")

# GOOD: Process row-by-row with .toLocalIterator()
# Memory-efficient for large datasets that must be processed in Python
row_count = 0
for row in data.toLocalIterator():
    row_count += 1
    if row_count % 10_000_000 == 0:
        print(f"Processed {row_count} rows")
        break  # Early exit possible

print(f"Successfully processed rows without driver OOM")

# 2. Rewrite Complex Joins as Broadcast Joins

In [5]:
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

# Large fact table: 10M transactions
transactions = spark.range(0, 10_000_000).withColumn(
    "user_id", (F.rand() * 10000).cast("int")
).withColumn("amount", F.rand() * 1000)

# Small dimension table: 1K users (150KB)
users = spark.range(0, 1000).withColumn(
    "user_name", F.concat(F.lit("user_"), F.col("id"))
)

# BAD: Shuffles both tables (slow)
slow_join = transactions.join(users, transactions.user_id == users.id)

# GOOD: Broadcast small table (fast)
fast_join = transactions.join(broadcast(users), transactions.user_id == users.id)

# Verify broadcast occurred
# fast_join.explain()  # Look for "BroadcastHashJoin"

# Force broadcast even if auto-detection fails
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")  # 10MB

# 3. Correct the Core/Task node Skew with SALTING

In [6]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

# Create skewed dataset: 95% of events from "popular_product"
skewed = spark.createDataFrame(
    [("popular_product", 1)] * 950_000 +
    [("rare_product", 1)] * 50_000,
    ["product_id", "sale_count"]
)

# Detect skew
skew_stats = skewed.groupBy("product_id").count().orderBy(F.col("count").desc())
skew_stats.show(truncate=False)  # popular_product: 950K, rare_product: 50K

# BAD: Single task processes all 950K popular_product records
# slow_agg = skewed.groupBy("product_id").sum("sale_count")

# GOOD: Salt the skewed key
N_SALT = 10  # Number of salt buckets

salted = skewed.withColumn(
    "salt",
    F.when(
        F.col("product_id") == "popular_product",
        (F.rand() * N_SALT).cast(IntegerType())
    ).otherwise(F.lit(None))  # Don't salt rare keys
).withColumn(
    "salted_key",
    F.concat(F.col("product_id"), F.lit("_"), F.col("salt"))
)

# First aggregation on salted key
first_agg = salted.groupBy("salted_key", "product_id").sum("sale_count")

# Second aggregation to combine salted results
final_result = first_agg.groupBy("product_id").sum("sale_count")

# Explain shows two aggregation stages instead of one giant task
final_result.explain()

print(f"Skew handled: {final_result.count()} final groups")

# 4. Cache Strategically (Not Everything)

In [7]:
from pyspark.storagelevel import StorageLevel

# Multi-stage pipeline: raw -> cleaned -> aggregated -> joined
raw_df = spark.range(0, 10_000_000).withColumn("value", F.rand() * 100)
# Stage 1: Cleaning (used by both aggregation paths)
cleaned_df = raw_df.filter(F.col("value") > 0.1).withColumn(
    "category", F.when(F.col("value") > 50, "high").otherwise("low")
)
# BAD: Caching everything
# cleaned_df.cache()  # Wastes memory if only used once
# GOOD: Cache ONLY if reused multiple times
# cleaned_df will be used in 2 downstream joins = cache-worthy
cleaned_df = cleaned_df.cache()
cleaned_df.count()  # Materialize cache
# Branch 1: Aggregation by category
agg_by_cat = cleaned_df.groupBy("category").agg(F.sum("value").alias("total_value"))
# Branch 2: Aggregation by value range
agg_by_range = cleaned_df.withColumn(
    "range", F.floor(F.col("value") / 10) * 10
).groupBy("range").agg(F.count("*").alias("count"))
# Final join (both branches depend on cleaned_df)
final_result = agg_by_cat.crossJoin(agg_by_range)
# Check if cache helped
# spark.sparkContext.uiWebUrl -> Storage tab shows cache usage
# Uncache when no longer needed
cleaned_df.unpersist()

# 5. Say Goodbye to UDFs: Use Built-In Functions


In [8]:
import time
from pyspark.sql.functions import udf, pandas_udf, PandasUDFType
import pandas as pd

df = spark.range(0, 1_000_000).withColumn("text", F.lit("hello world"))
# BAD: Python UDF (serializes each row)
@udf(returnType="string")
def slow_upper(text):
    return text.upper() if text else None
start = time.time()
result_udf = df.withColumn("upper_text", slow_upper(F.col("text")))
result_udf.count()
udf_time = time.time() - start
# GOOD: Built-in function (vectorized, JVM-native)
start = time.time()
result_builtin = df.withColumn("upper_text", F.upper(F.col("text")))
result_builtin.count()
builtin_time = time.time() - start
print(f"UDF time: {udf_time:.2f}s, Built-in time: {builtin_time:.2f}s")
print(f"Speedup: {udf_time/builtin_time:.0f}x")
# BETTER: Pandas UDF for complex logic (still faster than row UDF)
@pandas_udf(returnType="string")
def fast_upper(text: pd.Series) -> pd.Series:
    return text.str.upper()
result_pandas_udf = df.withColumn("upper_text", fast_upper(F.col("text")))
# Vectorized operations - multiple columns at once
vectorized_result = df.selectExpr(
    "upper(text) as upper_text",
    "length(text) as text_length",
    "concat(text, '_', id) as compound_key"
)

# 6. Partition Pruning: Read Only What You Need


In [9]:
from pyspark.sql import functions as F

# Write test data with partitioning
df = spark.range(0, 10_000_000).withColumn(
    "date", F.date_add(F.lit("2020-01-01"), (F.rand() * 365 * 3).cast("int"))
).withColumn("category", F.when(F.rand() > 0.5, "A").otherwise("B"))

# Partition by date (creates 1095 directories)
df.write.partitionBy("date").mode("overwrite").parquet("/tmp/partitioned_data")

# BAD: Reads all 1095 partitions, then filters
full_scan = spark.read.parquet("/tmp/partitioned_data")
result_slow = full_scan.filter(F.col("date") == "2021-06-15")

# GOOD: Prune at read time via partition filters
result_fast = spark.read.parquet("/tmp/partitioned_data")\
    .filter(F.col("date") == "2021-06-15")

# Verify pruning: Should show only 1 partition directory
print(f"Input files after pruning: {len(result_fast.inputFiles())}")
print(f"Input files without pruning: {len(full_scan.inputFiles())}")  # 1095

# Best: Push filter directly into path (most aggressive pruning)
best_result = spark.read.parquet("/tmp/partitioned_data/date=2021-06-15/")

# 7. Let Tungsten Do the Heavy Lifting

In [10]:
from pyspark.sql import functions as F

# Create 100M row dataset
df = spark.range(0, 100_000_000).withColumn("value", F.rand() * 1000)

# BAD: RDD-based processing (bypasses Tungsten)
rdd = df.rdd
def slow_process(row):
    return (row.id % 100, row.value * 2) if row.value > 500 else None
rdd_result = rdd.map(slow_process).filter(lambda x: x is not None).reduceByKey(lambda a,b: a+b)

# GOOD: DataFrame operations (Tungsten-optimized)
df_result = df.filter(F.col("value") > 500)\
    .withColumn("bucket", F.col("id") % 100)\
    .groupBy("bucket")\
    .agg(F.sum(F.col("value") * 2).alias("total"))

# Compare execution plans
print("RDD plan (no optimization available)")
print("DataFrame plan:")
df_result.explain(mode="formatted")  # Shows "WholeStageCodegen"

# Tungsten fuses filter+project+aggregate into single stage
# RDD requires 3 separate passes

# Access to internal binary format via unsafe row processing
# DataFrame operations work directly on Tungsten memory layout

# 8. Predicate Pushdown: Filter Early, Filter Often


In [11]:
from pyspark.sql import functions as F

# Write test Parquet with statistics
test_df = spark.range(0, 10_000_000).withColumns({
    "date": F.date_add(F.lit("2020-01-01"), (F.rand() * 1000).cast("int")),
    "status": F.when(F.rand() > 0.9, "active").otherwise("inactive"),
    "value": F.rand() * 1000
})
test_df.write.parquet("/tmp/pushdown_test")

# BAD: Filter after heavy operations (predicate pushed but data already loaded)
df_bad = spark.read.parquet("/tmp/pushdown_test")\
    .withColumn("expensive", F.col("value") * F.col("id"))\
    .filter(F.col("status") == "active")

# GOOD: Filter immediately (prunes entire row groups)
df_good = spark.read.parquet("/tmp/pushdown_test")\
    .filter(F.col("status") == "active")\
    .withColumn("expensive", F.col("value") * F.col("id"))

# Verify pushdown in plan
print("Bad plan:")
df_bad.explain()
print("\nGood plan:")
df_good.explain()  # Should show "PushedFilters: [IsNotNull(status), EqualTo(status,active)]"

# Best: Combine with partition discovery
df_best = spark.read.parquet("/tmp/pushdown_test")\
    .filter((F.col("date") >= "2022-01-01") & (F.col("status") == "active"))\
    .select("date", "value")  # Pushdown projection too

# Check actual data scanned
print(f"Files read: {len(df_best.inputFiles())}")

# 9. Coalesce Small Files Like a Pro

In [12]:
from pyspark.sql import functions as F

# Generate many small files (10M rows -> 1000 files by default)
large_df = spark.range(0, 10_000_000).repartition(1000)
large_df.write.mode("overwrite").parquet("/tmp/many_files")

# Check output file count
import os
file_count_before = len([f for f in os.listdir("/tmp/many_files") if f.endswith(".parquet")])
print(f"Files before: {file_count_before}")  # ~1000 files

# BAD: coalesce after writing (too late)
# df.write.parquet(...); spark.read.coalesce(50)  # Still reads 1000 files first

# GOOD: Repartition before write (balances size)
target_file_size_mb = 128
total_size_mb = 2000  # Estimate: 10M rows * ~200 bytes/row
target_partitions = max(total_size_mb // target_file_size_mb, 1)

optimized_df = large_df.repartition(target_partitions)
optimized_df.write.mode("overwrite").parquet("/tmp/few_files")

# Verify file count
file_count_after = len([f for f in os.listdir("/tmp/few_files") if f.endswith(".parquet")])
print(f"Files after repartition({target_partitions}): {file_count_after}")  # ~15 files

# Best practice: Adaptive coalescing for existing small files
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.initialPartitionNum", "1000")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "134217728")  # 128MB

# Read back and optimize
small_files_df = spark.read.parquet("/tmp/many_files")
small_files_df.write.mode("overwrite").parquet("/tmp/adaptive_coalesced")

# 10. Kill the Job If It is Gives Early Doom Signs


In [13]:
from pyspark.sql import SparkSession
import sys

# Aggressive timeouts for development environment
spark = SparkSession.builder \
    .appName("safe_job") \
    .config("spark.sql.broadcastTimeout", "300") \
    .config("spark.network.timeout", "120s") \
    .config("spark.executor.heartbeatInterval", "30s") \
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \
    .config("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "5") \
    .config("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "256MB") \
    .config("spark.task.maxFailures", "4") \
    .config("spark.stage.maxConsecutiveAttempts", "2") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

# Programmatic safety valve: Kill if shuffle too large
spark.sparkContext.setJobGroup("monitored_job", "Job with size limits")

def check_shuffle_size(df, max_shuffle_gb=100):
    """Terminate job if estimated shuffle exceeds limit"""
    # Force analysis
    plan = df._jdf.queryExecution().analyzed()
    stats = plan.stats()
    size_bytes = stats.sizeInBytes().value()

    shuffle_gb = size_bytes / (10243)
    if shuffle_gb > max_shuffle_gb:
        spark.sparkContext.cancelJobGroup("monitored_job")
        raise ValueError(
            f"Estimated shuffle size {shuffle_gb:.1f}GB exceeds {max_shuffle_gb}GB limit. "
            f"Add filters or increase salt buckets."
        )
    return df

# Example usage
risky_df = spark.range(0, 1_000_000_000).withColumn("skewed_key", F.lit("hot_key"))
try:
    safe_df = check_shuffle_size(risky_df)
    result = safe_df.groupBy("skewed_key").count()
    result.show()
except ValueError as e:
    print(f"Job killed early: {e}")
    sys.exit(1)

# Set up automated monitoring
# In production, integrate with SparkListener:
# spark.sparkContext.addPySparkListener(MyFailureDetector())